# pi0 — delta-action benchmark run (A100-80)

Replaces `train_pi0_v2_benchmark_runpod.ipynb`. Everything the 30k absolute-action
run got wrong or left on the table, in one place.

| | old run | this notebook |
|---|---|---|
| action space | absolute joint targets | **delta from state** (+ gripper absolute) — openpi's `make_bool_mask(6,-1)` |
| chunk sampling | every frame (0.06°/step) | **`ACTION_STRIDE`** frames (0.27° at 5; net gain w/ delta ~6.5×, measured) |
| LoRA targets | Gemma names only → SigLIP got q/k/v | **both towers** (`out_proj`, `fc1`, `fc2` added) |
| `torch.compile` | never enabled | **on** (`modeling_pi0.py:590` really calls it) |
| DataLoader | `num_workers=2`, no prefetch | **tuned + measured** before you commit |
| LR schedule | none in `common/train.py` | **warmup + cosine**, sqrt-scaled to batch |
| warm-start | n/a | **`--init-from`, stats buffers excluded** (the silent killer) |

Read `delta_joint/README.md` for *why*. The logic lives in `delta_joint/` and is
covered by 18 assertions — this notebook only wires it up, so it cannot drift from
the tested code the way the v2/v3 notebooks did.

## 0 · Pod setup

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/SreevaatsavB/fairino-fr5-policies.git"
REPO_DIR = Path("/workspace/fairino-fr5-act-pipeline")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

sys.path[:0] = [str(REPO_DIR / "common"), str(REPO_DIR / "delta_joint")]
os.chdir(REPO_DIR)
print(subprocess.run(["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
!pip -q install "lerobot==0.5.1" peft bitsandbytes wandb hf_transfer

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"{p.name}  {p.total_memory/1e9:.0f} GB  sm_{p.major}{p.minor}")

In [ ]:
import getpass
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"        # 10 GB pushes take minutes, not hours
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("HF token (hf_...): ").strip()
assert HF_TOKEN.startswith("hf_")
os.environ["HF_TOKEN"] = HF_TOKEN
from huggingface_hub import login, whoami
login(HF_TOKEN)
HF_USER = whoami()["name"]
print("logged in as", HF_USER)

## 1 · Parameters

Only `RUN_MODE`, `STEP_BUDGET` and `BATCH_SIZE` normally need touching.

**`RUN_MODE`** — `"probe"` runs the 3k-step A/B from the README (~3 h) so you can
compare fresh vs warm-start before committing 27 h. `"full"` runs the real budget.

In [ ]:
# ── what to run ───────────────────────────────────────────────────────────────
RUN_MODE      = "probe"          # "probe" (3k-step A/B) | "full"
INIT_FROM     = None             # None = fresh from pi0_base;
                                 # or a path/HF file to warm-start (weights only)

# ── the fix ───────────────────────────────────────────────────────────────────
DELTA         = True             # actions as offsets from state  (openpi: 6,-1)
ACTION_STRIDE = 5                # chunk samples every Kth frame  -> 5x the signal
FRAME_STRIDE  = 15               # chunk STARTS every Kth frame. Raised 5->15: with
                                 # ACTION_STRIDE=5 a chunk spans 8.3 s, so starts 5
                                 # frames apart overlap ~98%. Cuts the epoch 3x.

# ── data ──────────────────────────────────────────────────────────────────────
HF_DATASET_REPO = "Slifold/fr5-pick-place-lerobot-v2"
DATA_ROOT     = "/workspace/dataset"
CHUNK_SIZE    = 50
CAMERAS       = ["wrist_cam", "scene_cam"]
AUG_LEVEL     = "crops"
VAL_FRAC      = 0.05
TASK_TEXT     = "pick up the block and place it in the bin"

# ── model / finetune ──────────────────────────────────────────────────────────
POLICY        = "pi0"
PRETRAINED    = "lerobot/pi0_base"

# openpi's PRIMARY recipe is full fine-tune. Their README lists
#   Fine-Tuning (Full) > 70 GB  A100 (80GB)/H100     <- this pod
#   Fine-Tuning (LoRA) > 22.5 GB  RTX 4090
# and every LoRA config they ship is named *_low_mem_finetune. On an A100-80 there
# is no reason to run the 4090 recipe, so "full" is the default here.
#
#   full         everything trains (openpi's recipe). LORA_RANK ignored.
#   lora         adapters on Gemma; SigLIP + expert stay FULLY trainable, which is
#                openpi's actual shape (get_freeze_filter only freezes ".*llm.*",
#                and the model is nnx.Dict(llm=..., img=...) so img is never frozen)
#   expert_only  the 24 GB fallback. NOT an openpi recipe — no openpi SFT config
#                freezes the whole VLM. Use only if nothing else fits.
FINETUNE_MODE = "full"           # full | lora | expert_only
LORA_RANK     = 16               # used only when FINETUNE_MODE == "lora"
LORA_ALPHA    = 32
PROPRIO_MODE  = "full"           # full | dropout | none
COMPILE       = True
COMPILE_MODE  = "default"        # 'max-autotune' is lerobot's default and is the
                                 # one that had to be backed out on sm_100; try it
                                 # on A100 only after a default-mode run is green.

# ── optimisation ──────────────────────────────────────────────────────────────
BATCH_SIZE    = 32               # 48-64 should fit on A100-80 WITH grad ckpt on.
                                 # LR is sqrt-scaled automatically below.
LR            = None             # None -> scaled_lr(BATCH_SIZE) off openpi's 2.5e-5@32
WARMUP_STEPS  = 1000
LR_FLOOR      = 0.1              # cosine lands here x peak, not at 0
WEIGHT_DECAY  = 0.01
GRAD_CLIP     = 1.0
GRAD_CKPT     = True             # required: no-ckpt OOMs on 80 GB even at batch 24
SEED          = 42

# ── dataloader (common/train.py hardcodes num_workers=2 — that is the bottleneck)
NUM_WORKERS   = 8
PREFETCH      = 4

# ── logging / output ──────────────────────────────────────────────────────────
CKPT_DIR      = Path("/workspace/checkpoints_pi0_delta")
MODEL_REPO    = "auto"
USE_WANDB     = True
LOG_EVERY     = 50
VAL_EVERY     = 500              # sub-val, in optimizer steps
SAVE_EVERY    = 2000

STEP_BUDGET   = 3000 if RUN_MODE == "probe" else 50000
if RUN_MODE == "probe":
    WARMUP_STEPS = min(WARMUP_STEPS, 200)   # a 3k probe cannot spend 1k warming up

from speedups import scaled_lr, FULL_LORA_TARGETS, finetune_flags
LR = LR or scaled_lr(BATCH_SIZE)
FT = finetune_flags(FINETUNE_MODE, LORA_RANK)
print(f"finetune_mode={FINETUNE_MODE}  ->  {FT}")
print(f"RUN_MODE={RUN_MODE}  steps={STEP_BUDGET}  batch={BATCH_SIZE}  "
      f"lr={LR:.2e}  warmup={WARMUP_STEPS}")
print(f"delta={DELTA}  action_stride={ACTION_STRIDE}  frame_stride={FRAME_STRIDE}")
print(f"init_from={INIT_FROM}")

## 2 · Dataset

In [ ]:
from huggingface_hub import snapshot_download

if not Path(DATA_ROOT).exists():
    snapshot_download(HF_DATASET_REPO, repo_type="dataset", local_dir=DATA_ROOT)
print("dataset at", DATA_ROOT)

In [ ]:
from dataset import FR5Dataset
from dataset_delta import DeltaJointDataset

n_eps = int(FR5Dataset(DATA_ROOT, use_image=False).info["total_episodes"])
train_eps, val_eps = FR5Dataset.episode_split(n_eps, VAL_FRAC, SEED)

def build(eps, aug):
    return DeltaJointDataset(DATA_ROOT, CHUNK_SIZE, True, (224, 224),
                             episode_indices=eps, aug_level=aug,
                             frame_stride=FRAME_STRIDE,
                             action_stride=ACTION_STRIDE, delta=DELTA)

train_ds, val_ds = build(train_eps, AUG_LEVEL), build(val_eps, "none")
IMAGE_KEYS = [f"observation.images.{c}" for c in CAMERAS]
stats = train_ds.get_stats()

print(f"episodes  train {len(train_eps)}  val {len(val_eps)}")
print(f"samples   train {len(train_ds)}  val {len(val_ds)}")
print(f"action_space recorded: {train_ds.info['action_space']!r}")
print(f"action_std (joints):   {stats['action_std'][:6].round(3)}")
print(f"chunk spans {CHUNK_SIZE * ACTION_STRIDE / 30:.1f} s of robot time")

### Sanity: is the normaliser actually tighter?

If `action_std` here is not clearly below the absolute run's ~13°, the whole point
of this notebook has been lost — stop and find out why before burning GPU hours.

In [ ]:
import numpy as np
abs_std = FR5Dataset(DATA_ROOT, CHUNK_SIZE, False, episode_indices=train_eps,
                     frame_stride=FRAME_STRIDE).get_stats()["action_std"][:6].mean()
new_std = stats["action_std"][:6].mean()
print(f"absolute action_std {abs_std:6.3f} deg")
print(f"this run  action_std {new_std:6.3f} deg   -> {abs_std / new_std:.1f}x tighter")
assert new_std < abs_std, "normaliser did NOT tighten — check DELTA/ACTION_STRIDE"

## 3 · DataLoader — measure before you tune

`common/train.py` hardcodes `num_workers=2`. The v2 notebook measured **9.2 s/step
starved vs ~4.7 s/step compute-bound** — half the GPU's life spent waiting on JPEGs.

Compare `s/batch` below against your `s/step` in section 6. Large fraction → raise
`NUM_WORKERS`. **Do not raise `BATCH_SIZE` while you are input-bound**; it will not
help and it changes the LR, which confounds the A/B.

In [ ]:
from torch.utils.data import DataLoader
from speedups import loader_kwargs, probe_input_bound

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          **loader_kwargs(NUM_WORKERS, PREFETCH))
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          **loader_kwargs(max(2, NUM_WORKERS // 2), PREFETCH))

_ = probe_input_bound(train_loader, n=20)

## 4 · Model

In [ ]:
import importlib.util
from speedups import enable_compile, init_from

spec = importlib.util.spec_from_file_location(
    f"policy_{POLICY}", REPO_DIR / "policies" / POLICY / "model.py")
policy_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(policy_mod)

if COMPILE:
    enable_compile(policy_mod, COMPILE_MODE)
    # LoRA is injected AFTER PI0Policy.__init__ compiles forward/sample_actions,
    # so the first few steps recompile once as the guards fail. Expect a slow
    # start (~1-3 min); judge s/step only after step ~50.
    print(f"[compile] on, mode={COMPILE_MODE} (first steps recompile — ignore s/step early)")

CFG = {
    "dataset": {"root": DATA_ROOT, "chunk_size": CHUNK_SIZE, "use_image": True,
                "image_size": [224, 224], "camera_names": CAMERAS,
                "val_frac": VAL_FRAC, "aug_level": AUG_LEVEL,
                "frame_stride": FRAME_STRIDE},
    "model": {
        "state_dim": 7, "action_dim": 7, "pretrained": PRETRAINED,
        **FT,                                    # rank / freeze flags for the mode
        "vlm_lora_alpha": LORA_ALPHA, "vlm_lora_dropout": 0.05,
        # BOTH towers. The old list used Gemma names only, so SigLIP got q/k/v and
        # nothing else -> exactly the 207 pairs in the deploy gate log.
        "vlm_lora_targets": list(FULL_LORA_TARGETS),
        "paligemma_variant": "gemma_2b", "action_expert_variant": "gemma_300m",
        "max_state_dim": 32, "max_action_dim": 32, "tokenizer_max_length": 48,
        "num_inference_steps": 10, "n_action_steps": None,
        "dtype": "bfloat16", "gradient_checkpointing": GRAD_CKPT,
        "quantize": "none", "proprio_mode": PROPRIO_MODE,
        "proprio_dropout_rate": 0.3,
        "prompt_newline": True, "pad_resize": True,          # format v3
    },
    "training": {"batch_size": BATCH_SIZE, "lr": LR, "weight_decay": WEIGHT_DECAY,
                 "grad_clip": GRAD_CLIP, "seed": SEED, "max_steps": STEP_BUDGET,
                 "warmup_steps": WARMUP_STEPS, "device": "cuda"},
}

device = torch.device("cuda")
torch.manual_seed(SEED)
model = policy_mod.build_model(CFG, stats, device)

# LoRA mode only: replicate openpi's freeze shape. lerobot has no "freeze the LLM
# but not the vision tower" flag, so do it by name. NOT applied for full/expert_only.
if FINETUNE_MODE == "lora":
    from speedups import freeze_llm_keep_vision
    freeze_llm_keep_vision(model)

n_lora    = sum(p.numel() for n, p in model.named_parameters() if "lora_" in n)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
vis_train = sum(p.numel() for n, p in model.named_parameters()
                if "vision_tower" in n and p.requires_grad)
print(f"mode={FINETUNE_MODE}  LoRA {n_lora/1e6:.1f}M  trainable {trainable/1e6:.0f}M  "
      f"frozen {frozen/1e6:.0f}M")
print(f"vision tower trainable: {vis_train/1e6:.0f}M params")
# The camera rig is new to pi0_base, so a frozen vision tower is the single worst
# thing to freeze. openpi never does it — fail loudly rather than train blind.
assert vis_train > 0 or FINETUNE_MODE == "expert_only", \
    "vision tower is FROZEN — openpi's freeze filter never freezes it; check FINETUNE_MODE"

### Warm-start (optional)

`init_from` loads **weights only** and deliberately drops `action_mean` /
`action_std`. They are registered as *buffers*, so a plain `load_state_dict` would
restore the old **absolute** stats over the delta ones — targets collapse to 0.12
of a unit, the 5× gain is exactly cancelled, and the loss still goes down. Silent.

Optimizer state is never restored: Adam's moments are calibrated to the old scale.

In [ ]:
if INIT_FROM:
    before = model.action_std.clone()
    init_from(model, INIT_FROM, device=device)
    assert torch.allclose(model.action_std, before), "stats buffers leaked through!"
    print("verified: this run's delta stats survived the warm-start")
else:
    print("fresh from", PRETRAINED)

## 5 · Optimizer + schedule

`common/train.py` builds a bare AdamW with **no scheduler at all** — the
warmup+cosine the 30k run used lived only in the old notebook. It is wired here.

On warm-start do **not** lower the peak LR: the action expert sits in a basin built
around "copy the state", and leaving it is the whole point. A reduced LR glues it
in place and you would wrongly conclude warm-starting does not work.

In [ ]:
from speedups import warmup_cosine

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                              lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
scheduler = warmup_cosine(optimizer, STEP_BUDGET, WARMUP_STEPS, LR_FLOOR)
print(f"AdamW lr={LR:.2e} wd={WEIGHT_DECAY} | warmup {WARMUP_STEPS} -> cosine to "
      f"{LR*LR_FLOOR:.2e} over {STEP_BUDGET} steps")

## 6 · Train

Step-based, not epoch-based — with `FRAME_STRIDE` in play "epoch" is not a
comparable unit across runs. `val_l1` is the flow-matching MSE on **normalized**
actions.

**It will be higher than the old run's 0.0339, and that is correct.** The absolute
target contained a large trivially-predictable component (the current position);
removing it removes the easy win. Do not tune toward the old number — that number
was mostly measuring the shortcut.

In [ ]:
import csv, time, math

CKPT_DIR.mkdir(parents=True, exist_ok=True)
run = None
if USE_WANDB and os.environ.get("WANDB_API_KEY"):
    import wandb
    run = wandb.init(project="fr5-vla-benchmark",
                     name=f"pi0-delta{int(DELTA)}-as{ACTION_STRIDE}-bs{BATCH_SIZE}-{RUN_MODE}",
                     config={**CFG["model"], **CFG["training"],
                             "action_stride": ACTION_STRIDE, "delta": DELTA,
                             "frame_stride": FRAME_STRIDE, "init_from": str(INIT_FROM)})

metrics = CKPT_DIR / "metrics_steps.csv"
if not metrics.exists():
    metrics.write_text("step,train_l1,val_l1,lr,grad_norm,s_per_step\n")

@torch.no_grad()
def sub_val(n_batches=20):
    model.eval(); tot = k = 0
    for b in val_loader:
        obs = b["observation.state"].to(device)
        act = b["action"].to(device)
        pad = b["action_is_pad"].to(device)
        img = {k: b[k].to(device) for k in IMAGE_KEYS if k in b}
        _, l1, _ = model(obs, act, pad, img, task=list(b["task"]))
        tot += l1; k += 1
        if k >= n_batches:
            break
    model.train()
    return tot / max(k, 1)

model.train()
step, best, t0 = 0, float("inf"), time.perf_counter()
while step < STEP_BUDGET:
    for batch in train_loader:
        if step >= STEP_BUDGET:
            break
        obs = batch["observation.state"].to(device, non_blocking=True)
        act = batch["action"].to(device, non_blocking=True)
        pad = batch["action_is_pad"].to(device, non_blocking=True)
        img = {k: batch[k].to(device, non_blocking=True) for k in IMAGE_KEYS if k in batch}

        loss, l1, _ = model(obs, act, pad, img, task=list(batch["task"]))
        loss.backward()
        gn = torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
        optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)
        step += 1

        if step % LOG_EVERY == 0:
            sps = (time.perf_counter() - t0) / LOG_EVERY
            lr_now = optimizer.param_groups[0]["lr"]
            eta = (STEP_BUDGET - step) * sps / 3600
            print(f"step {step:6d}/{STEP_BUDGET}  l1={l1:.4f}  lr={lr_now:.2e}  "
                  f"gn={gn:.2f}  {sps:.2f}s/step  eta {eta:.1f}h")
            if run: run.log({"train_l1": l1, "lr": lr_now, "grad_norm": float(gn),
                             "s_per_step": sps}, step=step)
            t0 = time.perf_counter()

        if step % VAL_EVERY == 0:
            v = sub_val()
            print(f"          sub-val l1={v:.4f}" + ("  <- best" if v < best else ""))
            with open(metrics, "a") as f:
                csv.writer(f).writerow([step, f"{l1:.6f}", f"{v:.6f}",
                                        f"{optimizer.param_groups[0]['lr']:.3e}",
                                        f"{float(gn):.4f}", f"{sps:.3f}"])
            if run: run.log({"val_l1": v}, step=step)
            if v < best:
                best = v
                torch.save({"model_state": model.state_dict(), "config": CFG,
                            "stats": stats, "policy": POLICY, "epoch": step,
                            "val_l1": v,
                            "action_space": train_ds.info["action_space"]},
                           CKPT_DIR / "best.pt")
            t0 = time.perf_counter()

        if step % SAVE_EVERY == 0:
            torch.save({"model_state": model.state_dict(), "config": CFG,
                        "stats": stats, "policy": POLICY, "epoch": step,
                        "val_l1": best,
                        "action_space": train_ds.info["action_space"]},
                       CKPT_DIR / "last.pt")
            t0 = time.perf_counter()

print(f"done at step {step}, best sub-val l1 {best:.4f}")

## 7 · Push

`create_repo` first — `upload_file` does **not** create repos, and HF returns 404
for both "missing" and "your token cannot see it".

`PUSH_RESUME=False` by default: `last.pt` carries optimizer state (~2× the size)
and you should not reuse those moments across an action-space change anyway.

In [ ]:
from huggingface_hub import HfApi

PUSH_RESUME = False
repo = MODEL_REPO if MODEL_REPO != "auto" else f"{HF_USER}/fr5-pi0-delta-{RUN_MODE}"
files = [f for f in ["best.pt", "metrics_steps.csv"] + (["last.pt"] if PUSH_RESUME else [])
         if (CKPT_DIR / f).exists()]
gb = sum((CKPT_DIR / f).stat().st_size for f in files) / 1e9

card = f"""---
license: apache-2.0
tags: [robotics, vla, pi0, lerobot, fairino-fr5]
---
# {repo.split('/')[-1]}

- **action_space**: `{train_ds.info['action_space']}` — joint offsets from state,
  gripper absolute (openpi `make_bool_mask(6, -1)`); each chunk entry is
  {ACTION_STRIDE} frames apart and must be held {ACTION_STRIDE} control steps at deploy.
- **Deploy with** `python delta_joint/run.py deploy --checkpoint best.pt` — the plain
  `common/deploy.py` will NOT add the state back.
- Base `{PRETRAINED}` · LoRA r={LORA_RANK} on both towers · batch {BATCH_SIZE} ·
  lr {LR:.2e} (warmup {WARMUP_STEPS} + cosine) · {step} steps · best val_l1 {best:.4f}
- val_l1 is normalized-space MSE and is **not comparable** to absolute-action runs.
"""

print(f"pushing {files} (~{gb:.1f} GB) -> {repo}")
api = HfApi()
api.create_repo(repo, private=True, exist_ok=True)      # <- the line the old one missed
api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=repo)
api.upload_folder(folder_path=str(CKPT_DIR), repo_id=repo, allow_patterns=files)
print("done ->", f"https://huggingface.co/{repo}")

## 8 · Next

**If `RUN_MODE="probe"`** you now have one arm of the A/B. Run the other
(`INIT_FROM = None` vs the 30k checkpoint), compare `val_l1` curves in
`metrics_steps.csv`, set `RUN_MODE="full"` on the winner.

### Then, on the Linux GPU PC (robot side)

The same repo runs inference — clone it next to `so101-fr5-teleop`, same venv
recipe (`lerobot==0.5.1`). Everything below reads the recipe out of the
checkpoint (`delta_joint@5`), so it cannot be run at the wrong speed, without the
state added back, or with the int64-mask crash the training notebooks used to
patch by hand:

```bash
# 1. offline eval on held-out episodes (writes <ckpt_dir>/eval/ep*.npz)
python delta_joint/run.py eval --ckpt best.pt --episodes 4

# 2. the pre-registered pass/fail — exit 0 = book robot time, 1 = keep training
python delta_joint/gate.py <ckpt_dir>/eval

# 3. only if the gate passes: the robot
python delta_joint/run.py deploy --hf-repo <you>/fr5-pi0-delta-full --task "..."
```

Gate criteria (fixed in `delta_joint/gate.py` BEFORE the run, on purpose):
model beats the "don't move" baseline on every episode, and the predicted
gripper crosses 0.65 where the ground truth closes. The 30k run fails both
(1.656 vs 0.075 deg; gripper max 0.154). `val_l1` is deliberately not consulted.

Gate C from the old predeploy gate (instruction A/B) stays retired on this
dataset — see `delta_joint/README.md` §6.
